# Оценка ТЗ по меню виртуального ассистента

Запустите ячейки по порядку. Укажите `TZ_PATH` в папке `TZ` и при необходимости поменяйте модель.
Меню и пять примеров ТЗ из коммита `731b3ef` находятся в `evaluator/`; этот ноутбук находится в `evalutator/`.
Можно использовать собственную папку `evalutator/TZ`, изменив `TZ_DIR`.

Ключ `OPENAI_API_KEY` читается из `.env` в корне репозитория или `evalutator/.env` (приоритет у последнего).
Текст ТЗ передаётся в OpenAI API. Поддерживаются DOCX (включая таблицы), текстовые PDF, TXT и MD.
Сканы требуют предварительного OCR; рисунки и вложенные объекты DOCX автоматически не распознаются.

«Успешно проставлено» означает наличие одной из четырёх оценок, а не независимую проверку её правильности.
Уверенность — самооценка модели. Неясные требования помечаются «не определено».
Результаты требуют проверки аналитиком. Повторный запуск создаёт отдельную папку результатов.

In [ ]:
from pathlib import Path
_install_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (p / "evalutator/requirements.txt").is_file())
_requirements = str(_install_root / "evalutator/requirements.txt")
%pip install -r "{_requirements}"

Зависимости устанавливаются в текущее ядро. После первой установки при необходимости перезапустите ядро.

In [ ]:
from pathlib import Path
import sys
import os
from datetime import datetime, timezone
from uuid import uuid4

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "evalutator/effort_evaluator.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из корня репозитория или папки evalutator.")
sys.path.insert(0, str(ROOT / "evalutator"))
from effort_evaluator import load_menu, load_spec, evaluate, summarize, statistics, save_results, save_summary
from dotenv import dotenv_values
from openai import OpenAI
from IPython.display import display

# Настройки пользователя
MENU_PATH = ROOT / "evaluator/chatbot_virtual_assistant_effort_menu.csv"
TZ_DIR = ROOT / "evaluator/TZ"
TZ_PATH = TZ_DIR / "ТЗ - Сервис для создания виртуальных ассистентов (Electra 2.0).docx"
BATCH_SIZE = 7
MAX_SPEC_CHARS = 180_000  # Защита от переполнения контекста; ТЗ не обрезается.

config = {**dotenv_values(ROOT / ".env"), **dotenv_values(ROOT / "evalutator/.env")}
MODEL = config.get("OPENAI_EVALUATOR_MODEL") or config.get("OPENAI_LLM_MODEL") or "gpt-4.1-mini"
RUN_DIR = ROOT / "evalutator/results" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])

In [ ]:
menu = load_menu(MENU_PATH)
text = load_spec(TZ_PATH, TZ_DIR)
print(f"ТЗ: {TZ_PATH.name}; символов: {len(text):,}; пунктов меню: {len(menu)}; модель: {MODEL}")
display(menu)
if len(text) > MAX_SPEC_CHARS:
    raise ValueError("ТЗ превышает MAX_SPEC_CHARS. Увеличьте лимит с учётом контекста модели.")

In [ ]:
api_key = config.get("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Добавьте OPENAI_API_KEY в .env в корне репозитория или evalutator/.env.")
client = OpenAI(api_key=api_key, timeout=180.0, max_retries=3)
results = evaluate(client, MODEL, menu, text, batch_size=BATCH_SIZE, max_spec_chars=MAX_SPEC_CHARS)
# Сохраняем оценки ДО запроса заключения: ошибка второго этапа не потеряет разметку.
csv_path = save_results(results, RUN_DIR, TZ_PATH.stem)
print(f"Разметка сохранена: {csv_path}")
display(results)
display(__import__("pandas").DataFrame([statistics(results)]))

In [ ]:
summary = summarize(client, MODEL, menu, text, results)
additions, report_path = save_summary(summary, results, RUN_DIR, TZ_PATH.stem)
print("Общее заключение\n")
print(summary.conclusion)
print("\nВопросы для уточнения:")
for question in summary.questions:
    print("•", question)
print("\nРекомендуемые дополнения меню с критериями по категориям:")
if additions.empty:
    print("Дополнения не требуются.")
else:
    with __import__("pandas").option_context("display.max_colwidth", None):
        display(additions)
print(f"\nЗаключение: {report_path}")

Файлы сохраняются в `evalutator/results/<идентификатор запуска>/`:
- `*_оценка.csv`: **Блок, Компонент, Оценка, Уверенность, Коментарий** (написание последнего столбца сохранено по запросу).
- `*_заключение.txt`: заключение, статистика, вопросы и расшифровка дополнений.
- `*_дополнения_меню.csv`: новые компоненты с определениями отсутствует/small/medium/large.

CSV имеет кодировку UTF-8 с BOM для Excel. Результаты и `.env` исключены из Git.
При отказе API, ошибке структуры ответа, пропусках или повторах ID выполнение останавливается с ошибкой;
техническая ошибка не превращается в оценку «не определено». После ошибки заключения можно повторить только последнюю кодовую ячейку.